# Generate forensic interpretations

Run base or fine tuned Phi-4-mini on 35 unseen events; greedy decoding with up to 2048 new tokens.


In [ ]:
from pathlib import Path
import csv, json, gc, re, logging
from collections import Counter

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MAX_INTERPRETATIONS = 2
EXP = ROOT / 'models/phi4_mini/generation'
SPLITS = EXP / "splits"
ADAPTER_DIR = EXP / "adapters" / "adapter_full"
RESULTS = ROOT / 'results/phi4_mini/generation/raw'
MODEL_ID = "microsoft/Phi-4-mini-instruct"


RUN_MODE = "generate"
MODEL_VARIANT = "base"
NEW_EVENTS_CSV = ROOT / 'data/atomic_dataset/selected_events_atomic.csv'
INFER_LIMIT = 0
MAX_NEW_TOKENS = 2048
RUN_LABEL = "greedy_v1"
BLOCK_EXACT_OVERLAP = True


assert RUN_MODE in {"inspect", "generate"}
assert MODEL_VARIANT in {"finetuned", "base"}
assert INFER_LIMIT >= 0 and MAX_NEW_TOKENS >= 128
RELATIONS = [
    "xIntent", "xNeed", "xAttr", "xEffect", "xReact", "xWant",
    "oEffect", "oReact", "oWant", "isAfter", "HasSubEvent", "isBefore",
    "HinderedBy", "Causes", "xReason", "isFilledBy"
]
FIELDS = ["crime_category", "crime_subcategory", "brief_context"] + RELATIONS
TAXONOMY = {
    "Violent Crimes": ["Assault", "Homicide", "Kidnapping", "Robbery", "Sexual Offense"],
    "Property Crimes": ["Theft", "Burglary", "Arson", "Vandalism", "Trespassing"],
    "Financial Crimes": ["Fraud", "Money Laundering", "Embezzlement", "Bribery", "Tax Evasion"],
    "Cyber Crimes": ["Hacking", "Phishing", "Identity Theft", "Online Harassment"],
    "Organized Crimes": ["Drug Trafficking", "Human Trafficking", "Racketeering", "Weapon Smuggling"],
    "Public Order/Justice": ["Perjury", "Obstruction of Justice", "Disorderly Conduct"],
}
print("Experiment:", EXP)
print("Expected adapter:", ADAPTER_DIR)
print("Model:", MODEL_VARIANT, "| Mode:", RUN_MODE)


## Shared training prompt


In [ ]:
from models.phi4_mini.generation.prompt import SYSTEM_PROMPT, TRAIN_SYSTEM_PROMPT, get_user_prompt, user_prompt, messages_for


## Response schema


In [ ]:
import json
from typing import List, Optional, Any
from pydantic import BaseModel, Field, field_validator
import logging
log = logging.getLogger("fatomic_validator")


class AtomicEntry(BaseModel):
    event: str
    crime_category: str = Field(..., description="Primary category")
    crime_subcategory: str = Field(..., description="Specific subcategory")
    brief_context: Optional[str] = Field(None, description="Context needed for this interpretation")


    xIntent: List[str] = Field(default_factory=list)
    xNeed: List[str] = Field(default_factory=list)
    xAttr: List[str] = Field(default_factory=list)
    xEffect: List[str] = Field(default_factory=list)
    xReact: List[str] = Field(default_factory=list)
    xWant: List[str] = Field(default_factory=list)
    oEffect: List[str] = Field(default_factory=list)
    oReact: List[str] = Field(default_factory=list)
    oWant: List[str] = Field(default_factory=list)


    isAfter: List[str] = Field(default_factory=list)
    HasSubEvent: List[str] = Field(default_factory=list)
    isBefore: List[str] = Field(default_factory=list)
    HinderedBy: List[str] = Field(default_factory=list)
    Causes: List[str] = Field(default_factory=list)
    xReason: List[str] = Field(default_factory=list)
    isFilledBy: List[str] = Field(default_factory=list)

    @field_validator('crime_category')
    @classmethod
    def standardize_category(cls, v):
        """Standardize the Non-Criminal category"""
        v_lower = v.lower().strip()
        if v_lower in ["no crime", "not a crime", "innocent", "legal", "none"]:
            return "Non-Criminal"
        return v.strip()


    @field_validator(
        'xIntent', 'xWant', 'oReact', 'xReact', 'oWant', 'xEffect', 'oEffect', 'xNeed', 'xAttr',
        'isAfter', 'HasSubEvent', 'isBefore', 'HinderedBy', 'Causes', 'xReason', 'isFilledBy',
        mode='before'
    )
    @classmethod
    def force_list(cls, v: Any):
        """
        If the LLM returns an empty string or 'none' instead of a list,
        convert it to an empty list to prevent Pydantic crash.
        """
        if isinstance(v, str):
            v_clean = v.strip().lower()
            if v_clean == "" or v_clean in ["none", "null", "n/a"]:
                return []

            return [v]
        if v is None:
            return []
        return v

    @field_validator(
        'xIntent', 'xWant', 'oReact', 'xReact', 'oWant', 'xEffect', 'oEffect', 'xNeed', 'xAttr',
        'isAfter', 'HasSubEvent', 'isBefore', 'HinderedBy', 'Causes', 'xReason', 'isFilledBy'
    )
    @classmethod
    def check_content_quality(cls, v, info):
        """
        Validates that list items are not empty, not too short, and do not contain AI filler words.
        """
        if not v:
            return v


        cleaned = [item.strip() for item in v if len(item.strip()) >= 3]


        forbidden = [
            "i don't know", "depends on context", "as an ai", "unknown",
            "n/a", "not applicable", "i cannot", "ai model", "sorry", "illegal"
        ]

        final_list = []
        for item in cleaned:
            if not any(bad in item.lower() for bad in forbidden):
                final_list.append(item)

        return final_list


def validate_llm_output(json_data: Any, original_event: str) -> Optional[dict]:
    """
    Validates the raw dictionary from the LLM against the AtomicEntry schema.
    Returns the validated dictionary or None if validation fails.
    """


    if isinstance(json_data, str):
        if "not possible" in json_data.lower():
            log.warning(f"Impossible to create a criminal context for '{original_event}'")
            return None
        else:
            log.warning(f"Validation failed for '{original_event}': Expected dictionary, got string.")
            return None
    elif isinstance(json_data, list) and len(json_data) > 0 and isinstance(json_data[0], str) and "not possible" in json_data[0].lower():
        log.warning(f"Impossible to create a criminal context for '{original_event}'")
        return None


    if isinstance(json_data, list) and len(json_data) > 0 and isinstance(json_data[0], str):
        if "not possible" in json_data[0].lower():
            log.warning(f"Impossible to create a criminal context for '{original_event}'")
            return None


    if not isinstance(json_data, dict):
        log.warning(f"Validation failed for '{original_event}': Expected dictionary, got {type(json_data).__name__}")
        return None

    try:

        data_to_validate = json_data.copy()
        data_to_validate['event'] = original_event


        entry = AtomicEntry(**data_to_validate)
        return entry.model_dump()
    except Exception as e:
        log.warning(f"Validation failed for '{original_event}': {e}")
        return None


In [ ]:
def norm_event(s):
    return " ".join(str(s).casefold().split())

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def known_events():
    observed = set()
    for name in ("train", "val", "test"):
        file = SPLITS / f"{name}.jsonl"
        if not file.is_file():
            raise FileNotFoundError(f"Missing split: {file}. Use the splits in this experiment directory.")
        observed.update(norm_event(row["event"]) for row in read_jsonl(file))
    return observed

def inspect_training():
    if MODEL_VARIANT == "finetuned":
        required = ["adapter_model.safetensors", "adapter_config.json", "training_completed.json"]
        missing = [name for name in required if not (ADAPTER_DIR / name).is_file()]
        if missing:
            raise FileNotFoundError(f"Full fine-tuned adapter not found in {ADAPTER_DIR}. Missing: {missing}")
        info = json.loads((ADAPTER_DIR / "training_completed.json").read_text(encoding="utf-8"))
        print("Training complete:", {k: info.get(k) for k in
             ("model", "epochs", "max_seq_len", "max_interpretations", "optimizer_steps")})
        if info.get("model") != MODEL_ID or info.get("max_interpretations") != MAX_INTERPRETATIONS:
            raise ValueError("Training metadata specifies a different model or interpretation count.")
        prompt_saved = ADAPTER_DIR / "training_system_prompt.txt"
        if prompt_saved.exists() and prompt_saved.read_text(encoding="utf-8") != SYSTEM_PROMPT:
            raise ValueError("SYSTEM_PROMPT differs from the adapter copy; inference aborted.")
    known = known_events()
    print("Known F-ATOMIC events (train/validation/test):", len(known))
    return known

known = inspect_training()

if NEW_EVENTS_CSV.is_file():
    with NEW_EVENTS_CSV.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        print("External CSV found; columns:", reader.fieldnames)
        if "event" not in (reader.fieldnames or []):
            raise ValueError("The external CSV must contain an 'event' column.")
else:
    print("The external CSV is missing; prepare it before RUN_MODE='generate':", NEW_EVENTS_CSV)


## Validate generated responses


In [ ]:
def parse_output(raw_text, event):
    s = raw_text.strip()
    for token in ("<|end|>", "<|endoftext|>"):
        s = s.replace(token, "")
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.I).strip()
        s = re.sub(r"\s*```$", "", s).strip()
    if s.upper().strip('"') == "NOT POSSIBLE":
        return {"status": "not_possible", "parsed": "NOT POSSIBLE", "validated": [],
                "valid_strict": False, "issues": ["Risposta NOT POSSIBLE"]}
    try:
        data = json.loads(s)
    except json.JSONDecodeError as err:
        return {"status": "invalid_json", "parsed": None, "validated": [],
                "valid_strict": False, "issues": [str(err)]}
    if isinstance(data, str) and data.upper() == "NOT POSSIBLE":
        return {"status": "not_possible", "parsed": data, "validated": [],
                "valid_strict": False, "issues": ["Risposta NOT POSSIBLE"]}
    if not isinstance(data, list):
        return {"status": "invalid_schema", "parsed": data, "validated": [],
                "valid_strict": False, "issues": ["Output must be a JSON list"]}
    issues = []
    if not 1 <= len(data) <= 2:
        issues.append(f"Invalid interpretation count: {len(data)}")
    validated = []
    seen_pairs = set()
    for j, entry in enumerate(data):
        if not isinstance(entry, dict):
            issues.append(f"Interpretation {j}: expected a JSON object")
            continue
        missing = [k for k in FIELDS if k not in entry]
        if missing:
            issues.append(f"Interpretation {j}: missing fields: {missing}")
        extras = [k for k in entry if k not in FIELDS]
        if extras:
            issues.append(f"Interpretation {j}: unexpected fields: {extras}")
        cat, sub = entry.get("crime_category"), entry.get("crime_subcategory")
        if not isinstance(cat, str) or sub not in TAXONOMY.get(cat, []):
            issues.append(f"Interpretation {j}: pair outside taxonomy ({cat}, {sub})")
        if (cat, sub) in seen_pairs:
            issues.append(f"Interpretation {j}: coppia duplicata ({cat}, {sub})")
        seen_pairs.add((cat, sub))
        if not isinstance(entry.get("brief_context"), str) or not entry["brief_context"].strip():
            issues.append(f"Interpretation {j}: missing or empty brief_context")
        for rel in RELATIONS:
            v = entry.get(rel)
            if not isinstance(v, list) or not all(isinstance(x, str) and x.strip() for x in v):
                issues.append(f"Interpretation {j}: {rel} must be a list of nonempty strings")
        if "___" not in event and entry.get("isFilledBy") != []:
            issues.append(f"Interpretation {j}: isFilledBy should be [] without placeholder ___")
        clean = validate_llm_output(entry, event)
        if clean is None:
            issues.append(f"Interpretation {j}: rejected by the original validator")
        else:
            validated.append(clean)
    return {"status": "valid" if not issues else "invalid_schema", "parsed": data,
            "validated": validated, "valid_strict": not issues, "issues": issues}

print('Parser check:', parse_output('[{"crime_category":"Property Crimes"}]', 'PersonX steals ___')["status"])


## Load the selected model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| GiB:",
          round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_prompt(event):
    return tokenizer.apply_chat_template(messages_for(event), tokenize=False, add_generation_prompt=True)

def load_model():
    if not torch.cuda.is_available():
        raise RuntimeError("Use a Jupyter kernel with a CUDA GPU.")
    bf16 = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, torch_dtype=dtype,
        trust_remote_code=True, device_map={"": torch.cuda.current_device()})
    if MODEL_VARIANT == "finetuned":
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    model.eval()
    return model

def generate_one(model, event):
    prompt = render_prompt(event)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {key: value.to("cuda") for key, value in inputs.items()}
    eos_ids = [tokenizer.eos_token_id]
    end_id = tokenizer.convert_tokens_to_ids("<|end|>")
    if end_id in tokenizer.all_special_ids and end_id not in eos_ids:
        eos_ids.append(end_id)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    suffix = generated[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(suffix, skip_special_tokens=True)
    was_cut = len(suffix) >= MAX_NEW_TOKENS and int(suffix[-1]) not in eos_ids
    return raw, int(len(suffix)), bool(was_cut)


## Generate and save results


In [ ]:
def load_external():
    if not NEW_EVENTS_CSV.is_file():
        raise FileNotFoundError(f"New event CSV not found: {NEW_EVENTS_CSV}")
    with NEW_EVENTS_CSV.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if "event" not in (reader.fieldnames or []):
            raise ValueError("The CSV must contain an 'event' column")
        fields = list(reader.fieldnames)
        rows = list(reader)
    for i, row in enumerate(rows, 1):
        if not (row.get("event") or "").strip():
            raise ValueError(f"Empty event at CSV row {i+1}")
    overlaps = [(i, row["event"]) for i, row in enumerate(rows, 1)
                if norm_event(row["event"]) in known]
    print("External CSV events:", len(rows), "| exact F-ATOMIC matches:", len(overlaps))
    if overlaps:
        print("Overlap examples:", overlaps[:5])
        if BLOCK_EXACT_OVERLAP:
            raise ValueError("Events already occur in F-ATOMIC; this is not an unseen-event test.")
    return fields, rows

def out_paths():
    safe_stem = re.sub(r"[^A-Za-z0-9_-]+", "_", NEW_EVENTS_CSV.stem)[:60]
    safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", RUN_LABEL)[:50]
    stem = f"{MODEL_VARIANT}_{safe_stem}_{safe_label}_max{MAX_NEW_TOKENS}"
    return RESULTS / f"{stem}.jsonl", RESULTS / f"{stem}_events.csv", RESULTS / f"{stem}_interpretations.csv"

def export_reports(records, external_fields):
    _, path_events, path_interp = out_paths()
    event_fields = ["source_row", *external_fields, "model_variant", "generation_status",
                    "valid_strict", "num_parsed", "num_validated", "new_tokens", "hit_max_tokens",
                    "issues", "raw_response"]
    with path_events.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=event_fields, extrasaction="ignore")
        writer.writeheader()
        for row in records:
            writer.writerow({
                "source_row": row["source_row"], **row["input_row"],
                "model_variant": MODEL_VARIANT, "generation_status": row["status"],
                "valid_strict": row["valid_strict"],
                "num_parsed": len(row["parsed"]) if isinstance(row["parsed"], list) else 0,
                "num_validated": len(row["validated"]), "new_tokens": row["new_tokens"],
                "hit_max_tokens": row["hit_max_tokens"],
                "issues": json.dumps(row["issues"], ensure_ascii=False),
                "raw_response": row["raw_response"],
            })
    flat_fields = ["source_row", "branch_index", "event", *FIELDS]
    with path_interp.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=flat_fields, extrasaction="ignore")
        writer.writeheader()
        for row in records:
            if not row["valid_strict"]:
                continue
            for idx, branch in enumerate(row["validated"], 1):
                out = {"source_row": row["source_row"], "branch_index": idx,
                       "event": row["input_row"]["event"]}
                for field in FIELDS:
                    out[field] = (json.dumps(branch[field], ensure_ascii=False)
                                  if field in RELATIONS else branch.get(field, ""))
                writer.writerow(out)
    return path_events, path_interp

def generate_all():
    external_fields, all_rows = load_external()
    selected = all_rows if INFER_LIMIT == 0 else all_rows[:INFER_LIMIT]
    if not selected:
        raise ValueError("The new event CSV is empty")
    RESULTS.mkdir(parents=True, exist_ok=True)
    path_jsonl, _, _ = out_paths()
    past = read_jsonl(path_jsonl) if path_jsonl.is_file() else []
    done = {}
    for row in past:
        idx = row["source_row"]
        if idx in done:
            raise ValueError(f"Duplicate row in resume JSONL: {idx}")
        if idx > len(all_rows) or norm_event(row["input_row"]["event"]) != norm_event(all_rows[idx-1]["event"]):
            raise ValueError("CSV differs from the saved JSONL; change RUN_LABEL for a new experiment.")
        done[idx] = row
    print("Remaining to generate:", len(selected) - sum(i in done for i in range(1, len(selected) + 1)))
    if any(i not in done for i in range(1, len(selected)+1)):
        model = load_model()
        try:
            with path_jsonl.open("a", encoding="utf-8") as f:
                for i, input_row in enumerate(selected, 1):
                    if i in done:
                        continue
                    event = input_row["event"].strip()
                    raw, n_tokens, cut = generate_one(model, event)
                    parsed = parse_output(raw, event)
                    record = {"source_row": i, "input_row": input_row,
                              "model_variant": MODEL_VARIANT, "raw_response": raw,
                              "new_tokens": n_tokens, "hit_max_tokens": cut,
                              **parsed}
                    if cut:
                        record["issues"].append("Reached max_new_tokens; the response may be truncated")
                        record["valid_strict"] = False
                        record["status"] = "truncated"
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    done[i] = record
                    print(f"[{i}/{len(selected)}] {record['status']} | validated branches: {len(record['validated'])} | token: {n_tokens}")
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    ordered = [done[i] for i in range(1, len(selected)+1)]
    reports = export_reports(ordered, external_fields)
    print("Statuses:", dict(Counter(row["status"] for row in ordered)))
    print("Raw JSONL:", path_jsonl)
    print("Report CSV:", reports[0])
    print("Validated interpretations CSV:", reports[1])
    return path_jsonl, *reports


## Run mode


In [ ]:
if RUN_MODE == "inspect":
    print("Ready. Example prompt tokens:", len(tokenizer(render_prompt("PersonX cuts ___ in half"),
                                                     add_special_tokens=False)["input_ids"]))
    print("Original user prompt example:\n", get_user_prompt("PersonX cuts ___ in half"))
    print("To generate: use RUN_MODE='generate' and provide the CSV.")
else:
    generate_all()
